# B3 — So sánh bộ chỉ số đánh giá tự cài với BEIR

## Mục tiêu

Kiểm chứng tính đúng đắn của các chỉ số đánh giá truy xuất thông tin được tự cài đặt trong `chi_so.py` bằng cách đối chiếu với thư viện BEIR.

**Dữ liệu thực nghiệm:**
- SciFact: 300 truy vấn test.
- NFCorpus: 323 truy vấn test.
- Kết quả truy xuất Top-100 từ BM25 tự cài ở B2.
- Qrels test của từng bộ dữ liệu.

**Quy trình:**

1. Đọc kết quả truy xuất Top-100 và qrels.
2. Tính các chỉ số bằng `chi_so.py`.
3. Tính các chỉ số tương ứng bằng BEIR.
4. So sánh sai lệch và nhận xét kết quả.

Cả hai cách đánh giá sử dụng cùng dữ liệu đầu vào để bảo đảm tính công bằng.

In [3]:
import sys
import json
from pathlib import Path

# Thư mục gốc dự án
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import bộ chỉ số tự cài
import chi_so

# Import BEIR
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

# Đường dẫn dữ liệu
DATA_DIR = PROJECT_ROOT / "phanB" / "du_lieu"
RESULT_DIR = PROJECT_ROOT / "phanB" / "ket_qua"
print("Import thành công")
print("Thư mục dữ liệu:", DATA_DIR.relative_to(PROJECT_ROOT))
print("Thư mục kết quả:", RESULT_DIR.relative_to(PROJECT_ROOT))

Import thành công
Thư mục dữ liệu: phanB\du_lieu
Thư mục kết quả: phanB\ket_qua


## Đọc kết quả truy xuất và qrels

Sử dụng kết quả truy xuất Top-100 đã lưu từ B2 cho hai bộ dữ liệu SciFact và NFCorpus.

Dữ liệu cần thiết:
- `results`: Kết quả truy xuất, gồm mã truy vấn, mã tài liệu và điểm BM25.
- `qrels`: Nhãn mức độ liên quan giữa truy vấn và tài liệu.

Cả bộ chỉ số tự cài và BEIR sẽ sử dụng cùng `results` và `qrels` để bảo đảm việc so sánh công bằng.

In [4]:
# Đọc kết quả truy xuất Top-100
def load_results(filename):
    file_path = RESULT_DIR / filename

    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)


scifact_results = load_results("scifact_bm25_top100.json")
nfcorpus_results = load_results("nfcorpus_bm25_top100.json")


# Đọc qrels test
def load_qrels(dataset_name):
    dataset_dir = DATA_DIR / dataset_name

    _, _, qrels = GenericDataLoader(
        data_folder=str(dataset_dir)
    ).load(split="test")

    return qrels


scifact_qrels = load_qrels("scifact")
nfcorpus_qrels = load_qrels("nfcorpus")


# Kiểm tra dữ liệu
print("SciFact:")
print("Results:", len(scifact_results))
print("Qrels:", len(scifact_qrels))

print("\nNFCorpus:")
print("Results:", len(nfcorpus_results))
print("Qrels:", len(nfcorpus_qrels))

100%|██████████| 3633/3633 [00:00<00:00, 181024.36it/s]

SciFact:
Results: 300
Qrels: 300

NFCorpus:
Results: 323
Qrels: 323


## Tính các chỉ số đánh giá bằng BEIR

Sử dụng `EvaluateRetrieval` của BEIR để đánh giá kết quả truy xuất Top-100 của BM25 tự cài.

Các chỉ số được tính:
- `nDCG@10`: Chất lượng xếp hạng trong 10 tài liệu đầu tiên.
- `MAP@100`: Mean Average Precision tại 100 tài liệu.
- `Recall@100`: Tỷ lệ tài liệu liên quan được truy xuất trong Top-100.
- `Precision@10`: Tỷ lệ tài liệu liên quan trong Top-10.

BEIR tính các chỉ số dựa trên kết quả truy xuất và qrels test. Các giá trị này sẽ được dùng làm tham chiếu để kiểm chứng `chi_so.py`.

In [5]:
# Các mức k cần đánh giá
k_values = [10, 100]

def evaluate_beir(qrels, results):
    ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
        qrels,
        results,
        k_values
    )

    return {
        "nDCG@10": ndcg["NDCG@10"],
        "MAP@100": _map["MAP@100"],
        "Recall@100": recall["Recall@100"],
        "Precision@10": precision["P@10"]
    }


scifact_beir = evaluate_beir(scifact_qrels, scifact_results)
nfcorpus_beir = evaluate_beir(nfcorpus_qrels, nfcorpus_results)

print("SciFact:")
for metric, score in scifact_beir.items():
    print(f"{metric}: {score:.4f}")

print("\nNFCorpus:")
for metric, score in nfcorpus_beir.items():
    print(f"{metric}: {score:.4f}")

SciFact:
nDCG@10: 0.6830
MAP@100: 0.6403
Recall@100: 0.9282
Precision@10: 0.0903

NFCorpus:
nDCG@10: 0.3212
MAP@100: 0.1439
Recall@100: 0.2464
Precision@10: 0.2306


## Tính các chỉ số bằng bộ chỉ số tự cài

Sử dụng các hàm trong `chi_so.py` để đánh giá cùng kết quả truy xuất Top-100 đã sử dụng với BEIR.

Với mỗi truy vấn:
- Lấy danh sách mã tài liệu theo thứ tự xếp hạng.
- Lấy qrels tương ứng.
- Tính Precision@10, Recall@100, AP@100 và nDCG@10.

Sau đó lấy trung bình trên tất cả truy vấn để thu được các chỉ số của toàn bộ tập test.

In [6]:
def evaluate_custom(qrels, results):
    scores = {
        "nDCG@10": [],
        "MAP@100": [],
        "Recall@100": [],
        "Precision@10": []
    }

    for query_id, relevance in qrels.items():
        ranked_docs = results.get(query_id, {})
        ranking = list(ranked_docs.keys())

        scores["nDCG@10"].append(
            chi_so.ndcg_at_k(ranking, relevance, 10)
        )

        scores["MAP@100"].append(
            chi_so.average_precision(ranking[:100], relevance)
        )

        scores["Recall@100"].append(
            chi_so.recall_at_k(ranking, relevance, 100)
        )

        scores["Precision@10"].append(
            chi_so.precision_at_k(ranking, relevance, 10)
        )

    return {
        metric: sum(values) / len(values)
        for metric, values in scores.items()
    }


scifact_custom = evaluate_custom(scifact_qrels, scifact_results)
nfcorpus_custom = evaluate_custom(nfcorpus_qrels, nfcorpus_results)

for name, scores in [
    ("SciFact", scifact_custom),
    ("NFCorpus", nfcorpus_custom)
]:
    print(f"\n{name}:")
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")


SciFact:
nDCG@10: 0.6830
MAP@100: 0.6403
Recall@100: 0.9282
Precision@10: 0.0903

NFCorpus:
nDCG@10: 0.3219
MAP@100: 0.1442
Recall@100: 0.2464
Precision@10: 0.2307


## Đối chiếu kết quả đánh giá với BEIR

So sánh các chỉ số đánh giá tự cài với BEIR trên cùng kết quả truy xuất Top-100 và qrels test.

Sai lệch tuyệt đối được tính theo công thức:

\[
\Delta = |\text{Metric}_{\text{tự cài}} - \text{Metric}_{\text{BEIR}}|
\]

Kết quả được hiển thị với 6 chữ số thập phân để quan sát các sai lệch nhỏ.

Nếu xuất hiện sai lệch, cần kiểm tra định nghĩa chỉ số và cách xử lý kết quả xếp hạng trước khi kết luận về tính đúng đắn của bộ chỉ số tự cài.

In [7]:
def compare_metrics(dataset, custom, beir):
    print(f"\n{dataset}")
    print(f"{'Metric':<15} {'Tự cài':<12} {'BEIR':<12} {'Sai lệch'}")

    for metric in custom:
        custom_score = custom[metric]
        beir_score = beir[metric]
        difference = abs(custom_score - beir_score)

        print(
            f"{metric:<15} "
            f"{custom_score:<12.6f} "
            f"{beir_score:<12.6f} "
            f"{difference:.6f}"
        )


compare_metrics("SciFact", scifact_custom, scifact_beir)
compare_metrics("NFCorpus", nfcorpus_custom, nfcorpus_beir)


SciFact
Metric          Tự cài       BEIR         Sai lệch
nDCG@10         0.683030     0.683030     0.000000
MAP@100         0.640281     0.640280     0.000001
Recall@100      0.928222     0.928220     0.000002
Precision@10    0.090333     0.090330     0.000003

NFCorpus
Metric          Tự cài       BEIR         Sai lệch
nDCG@10         0.321883     0.321180     0.000703
MAP@100         0.144228     0.143890     0.000338
Recall@100      0.246407     0.246410     0.000003
Precision@10    0.230650     0.230650     0.000000


## Kiểm tra các tài liệu đồng điểm BM25

Khi nhiều tài liệu có cùng điểm BM25, thứ tự giữa chúng có thể khác nhau tùy cách xử lý của hệ thống đánh giá.

Chúng ta kiểm tra:
- Số truy vấn xuất hiện tài liệu đồng điểm.
- Số truy vấn có điểm BM25 bằng nhau tại ranh giới Top-10 hoặc Top-100.

Kết quả giúp xác định liệu thứ tự xếp hạng có thể là nguyên nhân gây sai lệch giữa bộ chỉ số tự cài và BEIR hay không.

In [8]:
def check_ties(results):
    queries_with_ties = 0
    ties_at_10 = 0
    ties_at_100 = 0

    for ranked_docs in results.values():
        scores = list(ranked_docs.values())

        if len(scores) != len(set(scores)):
            queries_with_ties += 1

        if len(scores) > 10 and scores[9] == scores[10]:
            ties_at_10 += 1

        if len(scores) > 100 and scores[99] == scores[100]:
            ties_at_100 += 1

    return queries_with_ties, ties_at_10, ties_at_100


for name, results in [
    ("SciFact", scifact_results),
    ("NFCorpus", nfcorpus_results)
]:
    total, top10, top100 = check_ties(results)

    print(f"\n{name}:")
    print("Truy vấn có tài liệu đồng điểm:", total)
    print("Đồng điểm tại ranh giới Top-10:", top10)
    print("Đồng điểm tại ranh giới Top-100:", top100)


SciFact:
Truy vấn có tài liệu đồng điểm: 57
Đồng điểm tại ranh giới Top-10: 1
Đồng điểm tại ranh giới Top-100: 0

NFCorpus:
Truy vấn có tài liệu đồng điểm: 220
Đồng điểm tại ranh giới Top-10: 6
Đồng điểm tại ranh giới Top-100: 0


## Kiểm tra cách BEIR xử lý tài liệu đồng điểm

Khi nhiều tài liệu có cùng điểm BM25, thứ tự xếp hạng có thể ảnh hưởng đến các chỉ số đánh giá.

Trong bước này, chúng ta kiểm tra mã nguồn hàm đánh giá của BEIR và thư viện pytrec_eval để xác định cách xử lý kết quả truy xuất.

Mục tiêu là tìm nguyên nhân sai lệch nDCG@10 và MAP@100 trên NFCorpus trước khi quyết định có cần điều chỉnh bộ chỉ số tự cài hay không.

In [9]:
import inspect
import pytrec_eval

print("BEIR EvaluateRetrieval.evaluate:")
print(inspect.getsource(EvaluateRetrieval.evaluate))

print("\nPhiên bản pytrec_eval:")
print(getattr(pytrec_eval, "__version__", "Không có thông tin phiên bản"))

BEIR EvaluateRetrieval.evaluate:
    @staticmethod
    def evaluate(
        qrels: dict[str, dict[str, int]],
        results: dict[str, dict[str, float]],
        k_values: list[int],
        ignore_identical_ids: bool = True,
    ) -> tuple[dict[str, float], dict[str, float], dict[str, float], dict[str, float]]:
        if ignore_identical_ids:
            logger.info(
                "For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this."
            )
            popped = []
            for qid, rels in results.items():
                for pid in list(rels):
                    if qid == pid:
                        results[qid].pop(pid)
                        popped.append(pid)

        ndcg = {}
        _map = {}
        recall = {}
        precision = {}

        for k in k_values:
            ndcg[f"NDCG@{k}"] = 0.0
            _map[f"MAP@{k}"] = 0.0
            recall[f"Recall@{k}"] = 0.0
  

## Kiểm tra ảnh hưởng của thứ tự tài liệu đồng điểm

BEIR sử dụng `pytrec_eval` để tính các chỉ số đánh giá.

Khi nhiều tài liệu có cùng điểm BM25, thứ tự của chúng trong kết quả truy xuất có thể khác với thứ tự mà `pytrec_eval` sử dụng.

Trong bước này, chúng ta so sánh nDCG@10 tính bằng hai cách cho từng truy vấn NFCorpus:

- Bộ chỉ số tự cài `chi_so.py`.
- Thư viện `pytrec_eval`.

Sau đó xác định số truy vấn có kết quả khác nhau và kiểm tra liệu chúng có chứa tài liệu đồng điểm hay không.

In [10]:
evaluator = pytrec_eval.RelevanceEvaluator(
    nfcorpus_qrels,
    {"ndcg_cut_10"}
)

trec_scores = evaluator.evaluate(nfcorpus_results)

different_queries = []

for query_id, relevance in nfcorpus_qrels.items():
    ranked_docs = nfcorpus_results[query_id]
    ranking = list(ranked_docs.keys())

    custom_score = chi_so.ndcg_at_k(ranking, relevance, 10)
    trec_score = trec_scores[query_id]["ndcg_cut_10"]

    if abs(custom_score - trec_score) > 1e-6:
        scores = list(ranked_docs.values())
        has_ties = len(scores) != len(set(scores))

        different_queries.append(
            (query_id, custom_score, trec_score, has_ties)
        )

print("Số truy vấn có nDCG@10 khác nhau:", len(different_queries))

print("\n5 truy vấn khác nhau đầu tiên:")
for query_id, custom, trec, has_ties in different_queries[:5]:
    print(
        f"Query: {query_id} | "
        f"Tự cài: {custom:.6f} | "
        f"pytrec_eval: {trec:.6f} | "
        f"Có đồng điểm: {has_ties}"
    )

Số truy vấn có nDCG@10 khác nhau: 5

5 truy vấn khác nhau đầu tiên:
Query: PLAIN-634 | Tự cài: 0.399569 | pytrec_eval: 0.318340 | Có đồng điểm: True
Query: PLAIN-934 | Tự cài: 0.787774 | pytrec_eval: 0.782739 | Có đồng điểm: True
Query: PLAIN-2660 | Tự cài: 0.249664 | pytrec_eval: 0.247031 | Có đồng điểm: True
Query: PLAIN-2770 | Tự cài: 0.568099 | pytrec_eval: 0.566739 | Có đồng điểm: True
Query: PLAIN-3131 | Tự cài: 0.510285 | pytrec_eval: 0.374488 | Có đồng điểm: True


## Kiểm tra thứ tự tài liệu đồng điểm

Kết quả đối chiếu cho thấy 5 truy vấn NFCorpus có nDCG@10 khác nhau giữa bộ chỉ số tự cài và `pytrec_eval`.

Cả 5 truy vấn đều chứa tài liệu có cùng điểm BM25.

Trong bước này, chúng ta kiểm tra truy vấn `PLAIN-634` bằng cách:

- Xem thứ tự Top-10 được lưu trong JSON.
- Kiểm tra các tài liệu có cùng điểm BM25.
- Thử thay đổi thứ tự tài liệu đồng điểm để quan sát ảnh hưởng đến nDCG@10.

Mục tiêu là xác định liệu sai lệch có xuất phát từ cách xử lý thứ tự tài liệu đồng điểm hay không.

In [11]:
query_id = "PLAIN-634"

ranked_docs = nfcorpus_results[query_id]
relevance = nfcorpus_qrels[query_id]

# Thứ tự gốc từ JSON
original_ranking = list(ranked_docs.keys())

# Thử sắp xếp theo điểm BM25 giảm dần,
# nếu đồng điểm thì sắp xếp theo doc_id
sorted_ranking = sorted(
    ranked_docs,
    key=lambda doc_id: (-ranked_docs[doc_id], doc_id)
)

print("Top-10 ban đầu:")
for i, doc_id in enumerate(original_ranking[:10], 1):
    print(
        i,
        doc_id,
        ranked_docs[doc_id],
        "rel =", relevance.get(doc_id, 0)
    )

print("\nTop-10 sau khi sắp xếp:")
for i, doc_id in enumerate(sorted_ranking[:10], 1):
    print(
        i,
        doc_id,
        ranked_docs[doc_id],
        "rel =", relevance.get(doc_id, 0)
    )

print("\nnDCG@10:")
print("Thứ tự gốc:",
      chi_so.ndcg_at_k(original_ranking, relevance, 10))

print("Thứ tự mới:",
      chi_so.ndcg_at_k(sorted_ranking, relevance, 10))

print("pytrec_eval:",
      trec_scores[query_id]["ndcg_cut_10"])

Top-10 ban đầu:
1 MED-4620 11.236337893175952 rel = 1
2 MED-5010 11.236337893175952 rel = 0
3 MED-4619 9.139278529016574 rel = 1
4 MED-5011 8.565903982112514 rel = 0
5 MED-861 7.832178713027872 rel = 0
6 MED-4303 7.7219396716059 rel = 0
7 MED-5009 7.603035430739539 rel = 0
8 MED-4618 7.54494600533 rel = 1
9 MED-1378 6.029441444673776 rel = 0
10 MED-3199 5.480650463826803 rel = 0

Top-10 sau khi sắp xếp:
1 MED-4620 11.236337893175952 rel = 1
2 MED-5010 11.236337893175952 rel = 0
3 MED-4619 9.139278529016574 rel = 1
4 MED-5011 8.565903982112514 rel = 0
5 MED-861 7.832178713027872 rel = 0
6 MED-4303 7.7219396716059 rel = 0
7 MED-5009 7.603035430739539 rel = 0
8 MED-4618 7.54494600533 rel = 1
9 MED-1378 6.029441444673776 rel = 0
10 MED-3199 5.480650463826803 rel = 0

nDCG@10:
Thứ tự gốc: 0.3995688713838975
Thứ tự mới: 0.3995688713838975
pytrec_eval: 0.31833954895937183


## Kiểm chứng ảnh hưởng của tài liệu đồng điểm

Truy vấn PLAIN-634 có hai tài liệu đứng đầu cùng điểm BM25 nhưng khác nhãn liên quan.

Thử hoán đổi vị trí hai tài liệu này và tính lại nDCG@10 để kiểm tra ảnh hưởng của thứ tự đồng điểm đến kết quả đánh giá.

In [12]:
query_id = "PLAIN-634"

ranking = list(nfcorpus_results[query_id].keys())
relevance = nfcorpus_qrels[query_id]

# Đảo vị trí hai tài liệu đồng điểm đầu tiên
swapped_ranking = ranking.copy()
swapped_ranking[0], swapped_ranking[1] = (
    swapped_ranking[1],
    swapped_ranking[0]
)

print("nDCG gốc:",
      chi_so.ndcg_at_k(ranking, relevance, 10))

print("nDCG sau khi đảo:",
      chi_so.ndcg_at_k(swapped_ranking, relevance, 10))

print("nDCG của BEIR:",
      trec_scores[query_id]["ndcg_cut_10"])

nDCG gốc: 0.3995688713838975
nDCG sau khi đảo: 0.31833954895937183
nDCG của BEIR: 0.31833954895937183


## Kiểm tra ảnh hưởng của tài liệu đồng điểm trên toàn bộ NFCorpus

Kết quả thử nghiệm trên PLAIN-634 cho thấy việc thay đổi thứ tự các tài liệu đồng điểm BM25 có thể làm thay đổi nDCG@10.

Tiếp tục kiểm tra toàn bộ NFCorpus bằng cách sắp xếp các tài liệu đồng điểm theo hai quy tắc khác nhau:

- Giữ nguyên thứ tự truy xuất ban đầu.
- Sắp xếp theo điểm BM25 giảm dần, sau đó theo mã tài liệu giảm dần.

So sánh kết quả với pytrec_eval để xác định mức độ ảnh hưởng của thứ tự đồng điểm.

Đây là phép thử để tìm hiểu cách xử lý đồng điểm, chưa phải thay đổi thuật toán BM25 hoặc công thức nDCG.

In [13]:
def evaluate_sorted_ndcg(results, qrels, k=10):
    scores = []

    for query_id, relevance in qrels.items():
        ranked_docs = results[query_id]

        # Sắp xếp doc_id giảm dần khi điểm BM25 bằng nhau
        ranking = sorted(
            ranked_docs,
            key=lambda doc_id: doc_id,
            reverse=True
        )

        ranking.sort(
            key=lambda doc_id: ranked_docs[doc_id],
            reverse=True
        )

        scores.append(
            chi_so.ndcg_at_k(ranking, relevance, k)
        )

    return sum(scores) / len(scores)


sorted_ndcg = evaluate_sorted_ndcg(
    nfcorpus_results,
    nfcorpus_qrels
)

print(f"Thứ tự gốc: {nfcorpus_custom['nDCG@10']:.6f}")
print(f"Sắp xếp đồng điểm: {sorted_ndcg:.6f}")
print(f"BEIR: {nfcorpus_beir['nDCG@10']:.6f}")

Thứ tự gốc: 0.321883
Sắp xếp đồng điểm: 0.321183
BEIR: 0.321180


## Đối chiếu các chỉ số sau khi xử lý đồng điểm

Kết quả thực nghiệm cho thấy việc sắp xếp các tài liệu đồng điểm BM25 theo mã tài liệu giảm dần giúp nDCG@10 gần như trùng với BEIR.

Trong bước này, áp dụng cùng quy tắc sắp xếp cho cả hai bộ dữ liệu và tính lại bốn chỉ số:

- nDCG@10
- MAP@100
- Recall@100
- Precision@10

Sau đó đối chiếu với BEIR để kiểm tra tính nhất quán của toàn bộ bộ chỉ số.

In [14]:
def sort_tied_results(results):
    sorted_results = {}

    for query_id, ranked_docs in results.items():
        ranking = sorted(ranked_docs, reverse=True)

        ranking.sort(
            key=lambda doc_id: ranked_docs[doc_id],
            reverse=True
        )

        sorted_results[query_id] = {
            doc_id: ranked_docs[doc_id]
            for doc_id in ranking
        }

    return sorted_results


# Chuẩn hóa thứ tự tài liệu đồng điểm
scifact_sorted = sort_tied_results(scifact_results)
nfcorpus_sorted = sort_tied_results(nfcorpus_results)

# Tính lại bằng bộ chỉ số tự cài
scifact_custom_sorted = evaluate_custom(
    scifact_qrels, scifact_sorted
)

nfcorpus_custom_sorted = evaluate_custom(
    nfcorpus_qrels, nfcorpus_sorted
)

# Đối chiếu với BEIR
compare_metrics(
    "SciFact",
    scifact_custom_sorted,
    scifact_beir
)

compare_metrics(
    "NFCorpus",
    nfcorpus_custom_sorted,
    nfcorpus_beir
)


SciFact
Metric          Tự cài       BEIR         Sai lệch
nDCG@10         0.683030     0.683030     0.000000
MAP@100         0.640280     0.640280     0.000000
Recall@100      0.928222     0.928220     0.000002
Precision@10    0.090333     0.090330     0.000003

NFCorpus
Metric          Tự cài       BEIR         Sai lệch
nDCG@10         0.321183     0.321180     0.000003
MAP@100         0.143887     0.143890     0.000003
Recall@100      0.246407     0.246410     0.000003
Precision@10    0.230650     0.230650     0.000000


## Kết luận đối chiếu bộ chỉ số

Đã đánh giá bốn chỉ số nDCG@10, MAP@100, Recall@100 và Precision@10 bằng bộ chỉ số tự cài và BEIR trên hai bộ dữ liệu SciFact và NFCorpus.

Kết quả cho thấy:

- SciFact: Sai lệch lớn nhất là 0.000003.
- NFCorpus: Sai lệch lớn nhất là 0.000003.
- Tất cả các chỉ số đều khớp với BEIR trong sai số làm tròn.

Trong quá trình đối chiếu, phát hiện một số tài liệu có cùng điểm BM25 nhưng khác nhãn liên quan, dẫn đến sai lệch thứ hạng và kết quả nDCG, MAP.

Thử nghiệm sắp xếp các tài liệu đồng điểm theo doc_id giảm dần giúp tái tạo gần như chính xác kết quả đánh giá của BEIR.

**Kết luận:** Bộ chỉ số tự cài cho kết quả nhất quán với BEIR trên cả hai bộ dữ liệu được kiểm thử. Phần B3 hoàn thành.

In [16]:
for name, qrels, results in [
    ("SciFact", scifact_qrels, scifact_results),
    ("NFCorpus", nfcorpus_qrels, nfcorpus_results)
]:
    recall_10 = sum(
        chi_so.recall_at_k(
            list(results[qid].keys()), relevance, 10
        )
        for qid, relevance in qrels.items()
    ) / len(qrels)

    recall_100 = sum(
        chi_so.recall_at_k(
            list(results[qid].keys()), relevance, 100
        )
        for qid, relevance in qrels.items()
    ) / len(qrels)

    print(f"{name}:")
    print(f"  Recall@10:  {recall_10:.6f}")
    print(f"  Recall@100: {recall_100:.6f}")
    print(f"  PASS: {recall_100 > recall_10}")

SciFact:
  Recall@10:  0.817278
  Recall@100: 0.928222
  PASS: True
NFCorpus:
  Recall@10:  0.153071
  Recall@100: 0.246407
  PASS: True
